# DATA 422 Team 2

## Connection to NanoChat and the Hundred-Page Language Models Book

This track separates **architecture** from **pretrained weights**. Like Karpathy's NanoChat teaching pipeline, we initialize a causal Transformer from random weights and train next-token prediction. Unlike a general language model, we deliberately make the task narrow: structured evidence in, short constrained report out.

The Hundred-Page Language Models Book emphasizes tokenization, causal next-token training, Transformers, and post-training. Here we shrink the problem aggressively so a tiny decoder has a chance to learn the report grammar.

## 0. Colab setup

In [ ]:
%pip -q install -U "transformers>=4.56,<6" torch pandas matplotlib

In [ ]:
from pathlib import Path
import json, re, time, math, random
import numpy as np
import pandas as pd

SEED = 422
random.seed(SEED)
np.random.seed(SEED)

DEV_JSONL = Path("report_evidence_development.jsonl")
DEV_REPORTS = Path("reference_reports_development.csv")
TEST_JSONL = Path("report_evidence_test.jsonl")
GLOBAL_JSON = Path("global_model_information.json")

# The real project files will be used automatically when present.
# Until then, this small fallback makes the instructor notebook runnable.
FALLBACK_PACKETS = [
    {
        "case_id":"D001","region":"Northern Interior",
        "public_model":{"prediction":0.31,"base_value":0.42,
            "top_positive":[{"feature":"slope_deg","value":26.0,"shap":0.07}],
            "top_negative":[{"feature":"wrc_burn_probability","value":0.08,"shap":-0.11}]},
        "company_model":{"prediction":0.34,"base_value":0.45,
            "top_positive":[{"feature":"roof_age_years","value":32,"shap":0.09}],
            "top_negative":[{"feature":"defensible_space_ft","value":95,"shap":-0.13}]},
        "comparison":{"absolute_difference":0.03,"agreement_category":"close agreement","higher_model":"company"},
        "inputs":{"slope_deg":26.0,"wrc_burn_probability":0.08,"roof_age_years":32,"defensible_space_ft":95}
    },
    {
        "case_id":"D002","region":"Southern California",
        "public_model":{"prediction":0.43,"base_value":0.42,
            "top_positive":[{"feature":"fhsz_category","value":"Very High","shap":0.16},{"feature":"slope_deg","value":31.0,"shap":0.08}],
            "top_negative":[{"feature":"elevation_m","value":155,"shap":-0.03}]},
        "company_model":{"prediction":0.78,"base_value":0.45,
            "top_positive":[{"feature":"roof_age_years","value":41,"shap":0.19},{"feature":"defensible_space_ft","value":18,"shap":0.17}],
            "top_negative":[{"feature":"structure_year_built","value":1998,"shap":-0.04}]},
        "comparison":{"absolute_difference":0.35,"agreement_category":"substantial disagreement","higher_model":"company"},
        "inputs":{"fhsz_category":"Very High","slope_deg":31.0,"elevation_m":155,"roof_age_years":41,"defensible_space_ft":18,"structure_year_built":1998}
    },
    {
        "case_id":"D003","region":"North Coast / Bay",
        "public_model":{"prediction":0.72,"base_value":0.42,
            "top_positive":[{"feature":"wrc_burn_probability","value":0.31,"shap":0.18}],
            "top_negative":[{"feature":"fire_distance_km","value":6.2,"shap":-0.05}]},
        "company_model":{"prediction":0.69,"base_value":0.45,
            "top_positive":[{"feature":"canopy_cover","value":0.64,"shap":0.15}],
            "top_negative":[{"feature":"roof_class","value":"Class A","shap":-0.08}]},
        "comparison":{"absolute_difference":0.03,"agreement_category":"close agreement","higher_model":"public"},
        "inputs":{"wrc_burn_probability":0.31,"fire_distance_km":6.2,"canopy_cover":0.64,"roof_class":"Class A"}
    }
]

GLOBAL_FALLBACK = {
    "public_model":{"name":"California Public Model Proxy (logistic regression)","roc_auc":0.78,"brier":0.19,
        "permutation_importance":["wrc_burn_probability","fhsz_category","slope_deg"]},
    "company_model":{"name":"Company Internal Model (GBDT)","roc_auc":0.84,"brier":0.16,
        "permutation_importance":["defensible_space_ft","roof_age_years","canopy_cover"]}
}

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def load_project_data():
    dev = load_jsonl(DEV_JSONL) if DEV_JSONL.exists() else FALLBACK_PACKETS
    test = load_jsonl(TEST_JSONL) if TEST_JSONL.exists() else FALLBACK_PACKETS
    global_info = json.loads(GLOBAL_JSON.read_text(encoding='utf-8')) if GLOBAL_JSON.exists() else GLOBAL_FALLBACK
    refs = pd.read_csv(DEV_REPORTS) if DEV_REPORTS.exists() else None
    return dev, test, global_info, refs

dev_packets, test_packets, global_info, reference_reports = load_project_data()
print(f"Development packets: {len(dev_packets)}")
print(f"Test packets: {len(test_packets)}")
print("Reference reports available:", reference_reports is not None)

## 1. Normalize evidence

In [ ]:
# ---- Evidence-packet adapter -------------------------------------------------
# When Cursor finishes the real evidence schema, modify ONLY this function if
# field names differ. The rest of the notebook uses the normalized representation.

def _contributors(model_block, key):
    out = []
    for item in model_block.get(key, []) or []:
        if isinstance(item, dict):
            out.append({
                "feature": str(item.get("feature", "")),
                "value": item.get("value", None),
                "contribution": float(item.get("shap", item.get("contribution", 0.0)))
            })
    return out

def normalize_packet(packet):
    pub = packet.get("public_model", {})
    comp = packet.get("company_model", {})
    cmp = packet.get("comparison", {})
    return {
        "case_id": str(packet.get("case_id", "")),
        "region": str(packet.get("region", "Unknown")),
        "public_prediction": float(pub.get("prediction")),
        "company_prediction": float(comp.get("prediction")),
        "public_positive": _contributors(pub, "top_positive"),
        "public_negative": _contributors(pub, "top_negative"),
        "company_positive": _contributors(comp, "top_positive"),
        "company_negative": _contributors(comp, "top_negative"),
        "absolute_difference": float(cmp.get("absolute_difference", abs(float(pub.get("prediction"))-float(comp.get("prediction"))))),
        "agreement_category": str(cmp.get("agreement_category", "not supplied")),
        "higher_model": str(cmp.get("higher_model", "not supplied")),
        "inputs": packet.get("inputs", {}),
    }

normalized_dev = [normalize_packet(p) for p in dev_packets]
normalized_test = [normalize_packet(p) for p in test_packets]
pd.DataFrame(normalized_dev)[["case_id","region","public_prediction","company_prediction","absolute_difference","agreement_category"]].head()

In [ ]:
def fmt_contrib(items, max_n=3):
    if not items:
        return "none supplied"
    parts=[]
    for x in items[:max_n]:
        val = x.get("value")
        val_text = f"={val}" if val is not None else ""
        parts.append(f"{x['feature']}{val_text} (SHAP {x['contribution']:+.3f})")
    return "; ".join(parts)

def evidence_text(p):
    return f"""CASE {p['case_id']} | REGION: {p['region']}

PUBLIC MODEL PROXY
Prediction: {p['public_prediction']:.3f}
Positive local contributors: {fmt_contrib(p['public_positive'])}
Negative local contributors: {fmt_contrib(p['public_negative'])}

COMPANY INTERNAL MODEL
Prediction: {p['company_prediction']:.3f}
Positive local contributors: {fmt_contrib(p['company_positive'])}
Negative local contributors: {fmt_contrib(p['company_negative'])}

MODEL COMPARISON
Absolute prediction difference: {p['absolute_difference']:.3f}
Agreement category: {p['agreement_category']}
Higher prediction: {p['higher_model']}
"""

print(evidence_text(normalized_dev[0]))

## 2. Canonicalize the input

Task design can compensate for lack of pretraining. We do not ask a randomly initialized model to understand arbitrary prose. We convert every packet into a compact, stable representation that preserves the same evidence.

In [ ]:
def canonical_input(p):
    def f(items):
        return '|'.join(x['feature'] for x in items[:2]) if items else 'NONE'
    return (
        f"CASE={p['case_id']};REGION={p['region']};"
        f"PUB={p['public_prediction']:.2f};COMP={p['company_prediction']:.2f};"
        f"AGREE={p['agreement_category']};"
        f"PUB_UP={f(p['public_positive'])};PUB_DOWN={f(p['public_negative'])};"
        f"COMP_UP={f(p['company_positive'])};COMP_DOWN={f(p['company_negative'])};"
    )
print(canonical_input(normalized_dev[1]))

## 3. Create supervised targets

When the real `reference_reports_development.csv` is present, use its validated reports. The fallback below creates canonical targets so this instructor notebook can run before the real corpus arrives.

In [ ]:
def fallback_target(p):
    return (
        f"The public model proxy estimates serious-damage risk at {p['public_prediction']:.2f}, while the company model estimates {p['company_prediction']:.2f}. "
        f"The models show {p['agreement_category']}. "
        f"For the public proxy, important upward contributors include {', '.join(x['feature'] for x in p['public_positive'][:2]) or 'none supplied'}, and downward contributors include {', '.join(x['feature'] for x in p['public_negative'][:2]) or 'none supplied'}. "
        f"For the company model, important upward contributors include {', '.join(x['feature'] for x in p['company_positive'][:2]) or 'none supplied'}, and downward contributors include {', '.join(x['feature'] for x in p['company_negative'][:2]) or 'none supplied'}. "
        "These attributions describe model behavior, not causal effects, and the report does not make an underwriting decision."
    )

if reference_reports is not None and {'case_id','reference_report'}.issubset(reference_reports.columns):
    ref_map=dict(zip(reference_reports.case_id.astype(str), reference_reports.reference_report.astype(str)))
else:
    ref_map={p['case_id']:fallback_target(p) for p in normalized_dev}

pairs=[(canonical_input(p),ref_map[p['case_id']]) for p in normalized_dev if p['case_id'] in ref_map]
print(pairs[0][0]); print('\nTARGET:\n',pairs[0][1])

## 4. Byte tokenizer

For a tiny from-scratch model, a byte vocabulary avoids spending the whole assignment learning a tokenizer. It is inefficient compared with modern subword tokenization, but completely deterministic and able to represent every character.

In [ ]:
class ByteTokenizer:
    PAD=256; BOS=257; EOS=258
    vocab_size=259
    def encode(self,s): return [self.BOS]+list(s.encode('utf-8'))+[self.EOS]
    def decode(self,ids):
        bs=bytes([i for i in ids if 0<=i<256])
        return bs.decode('utf-8',errors='ignore')
tok=ByteTokenizer()
print(tok.decode(tok.encode('risk=0.72')))

## 5. Tiny NanoChat model from random weights

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import NanoChatConfig, NanoChatForCausalLM

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def make_model(layers=4, hidden=128, heads=4, kv_heads=2, max_len=1024):
    cfg=NanoChatConfig(vocab_size=tok.vocab_size,hidden_size=hidden,intermediate_size=4*hidden,
        num_hidden_layers=layers,num_attention_heads=heads,num_key_value_heads=kv_heads,
        max_position_embeddings=max_len,bos_token_id=tok.BOS,eos_token_id=tok.EOS,pad_token_id=tok.PAD,
        tie_word_embeddings=True,use_cache=True)
    return NanoChatForCausalLM(cfg)
model=make_model().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Completion-only training examples

The model receives `INPUT ... REPORT` and loss is applied only to report tokens. This parallels response-only supervised fine-tuning: we care about learning the desired output, not predicting the evidence prefix itself.

In [ ]:
def make_example(inp,target,max_len=1024):
    prefix=f"INPUT\n{inp}\nREPORT\n"
    a=tok.encode(prefix)[:-1]
    b=tok.encode(target)
    ids=(a+b)[:max_len]
    labels=[-100]*min(len(a),len(ids)) + ids[len(a):]
    return torch.tensor(ids),torch.tensor(labels)

class PairDS(Dataset):
    def __init__(self,pairs): self.examples=[make_example(*p) for p in pairs]
    def __len__(self): return len(self.examples)
    def __getitem__(self,i): return self.examples[i]

def collate(batch):
    L=max(len(x[0]) for x in batch)
    ids=torch.full((len(batch),L),tok.PAD,dtype=torch.long)
    labels=torch.full((len(batch),L),-100,dtype=torch.long)
    mask=torch.zeros((len(batch),L),dtype=torch.long)
    for i,(x,y) in enumerate(batch):
        ids[i,:len(x)]=x; labels[i,:len(y)]=y; mask[i,:len(x)]=1
    return {'input_ids':ids,'attention_mask':mask,'labels':labels}

## 7. Train/validation split and training loop

In [ ]:
# With the real corpus, keep a fixed validation subset separate from final test packets.
rng=np.random.default_rng(SEED); idx=np.arange(len(pairs)); rng.shuffle(idx)
cut=max(1,int(.8*len(idx))); train_pairs=[pairs[i] for i in idx[:cut]]; val_pairs=[pairs[i] for i in idx[cut:]] or train_pairs[-1:]
train_loader=DataLoader(PairDS(train_pairs),batch_size=min(8,len(train_pairs)),shuffle=True,collate_fn=collate)
val_loader=DataLoader(PairDS(val_pairs),batch_size=min(8,len(val_pairs)),shuffle=False,collate_fn=collate)

_model_load_start=time.perf_counter()
model=make_model().to(device)
if device.type=='cuda': torch.cuda.synchronize()
model_load_seconds=time.perf_counter()-_model_load_start
opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=.01)

def eval_loss(loader):
    model.eval(); vals=[]
    with torch.no_grad():
        for b in loader:
            b={k:v.to(device) for k,v in b.items()}; vals.append(model(**b).loss.item())
    return float(np.mean(vals))

if device.type=='cuda': torch.cuda.synchronize()
_training_start=time.perf_counter()
history=[]
for epoch in range(30 if len(pairs)<50 else 8):
    model.train(); losses=[]
    for b in train_loader:
        b={k:v.to(device) for k,v in b.items()}; opt.zero_grad(); out=model(**b); out.loss.backward();
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); losses.append(out.loss.item())
    vl=eval_loss(val_loader); history.append((epoch+1,float(np.mean(losses)),vl))
    if epoch%2==0: print(history[-1])
if device.type=='cuda': torch.cuda.synchronize()
training_seconds=time.perf_counter()-_training_start
print(f'Model construction/load time: {model_load_seconds:.3f} s')
print(f'Training time: {training_seconds:.3f} s')


## 8. Plot learning

In [ ]:
import matplotlib.pyplot as plt
h=pd.DataFrame(history,columns=['epoch','train_loss','val_loss'])
h.plot(x='epoch',y=['train_loss','val_loss']); plt.ylabel('cross-entropy loss'); plt.show()

## 9. Generate reports

In [ ]:
@torch.inference_mode()
def generate_tiny(p,max_new_tokens=220):
    model.eval(); prefix=f"INPUT\n{canonical_input(p)}\nREPORT\n"; ids=torch.tensor([tok.encode(prefix)[:-1]],device=device)
    out=model.generate(ids,max_new_tokens=max_new_tokens,do_sample=False,eos_token_id=tok.EOS,pad_token_id=tok.PAD,use_cache=True)
    gen=out[0,len(ids[0]):].tolist(); return tok.decode(gen).strip()
print(generate_tiny(normalized_dev[0]))

## 10. Audit against evidence

In [ ]:
FORBIDDEN_PATTERNS = [
    r"\bapprove\b", r"\bdeny\b", r"\bcancel\b", r"\bpremium should\b",
    r"\bwill burn\b", r"\bcauses? wildfire\b", r"\bguarantee(?:d|s)?\b"
]

def audit_report(report, p):
    text = report.lower()
    checks = {}
    # Numeric preservation: accept 0.43 or 43% style representations.
    def contains_prob(v):
        return (f"{v:.2f}" in report) or (f"{100*v:.0f}%" in report)
    checks["public_prediction"] = contains_prob(p["public_prediction"])
    checks["company_prediction"] = contains_prob(p["company_prediction"])
    checks["agreement"] = p["agreement_category"].lower() in text
    # Allow reports to summarize a subset of contributors, but any named canonical top feature should be traceable.
    allowed_features = {x['feature'].lower() for k in ["public_positive","public_negative","company_positive","company_negative"] for x in p[k]}
    checks["no_forbidden_action_or_causal_claim"] = not any(re.search(pattern, text) for pattern in FORBIDDEN_PATTERNS)
    checks["concise"] = len(report.split()) <= 220
    return checks

def audit_score(report, p):
    c = audit_report(report, p)
    return sum(c.values())/len(c), c

In [ ]:
for p in normalized_dev[:5]:
    r=generate_tiny(p); print('\n',p['case_id'],audit_score(r,p)); print(r)

## 11. Smart improvement experiments

Try only a few changes: (a) 4 vs 6 layers, (b) hidden size 128 vs 192, (c) canonicalized input versus a more natural-language input, or (d) more/less training data. This mirrors NanoChat's educational philosophy: understand scaling and training behavior rather than blindly searching hyperparameters.

## 12. Official held-out outputs

In [ ]:
rows=[]
for p in normalized_test:
    if device.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter(); r=generate_tiny(p)
    if device.type=='cuda': torch.cuda.synchronize()
    dt=time.perf_counter()-t0; s,c=audit_score(r,p)
    rows.append({'case_id':p['case_id'],'report':r,'seconds':dt,'audit_score':s,**c})
pd.DataFrame(rows).to_csv('team_B_test_reports.csv',index=False)
print('saved team_B_test_reports.csv')

## 13. Save the trained tiny model

In [ ]:
model.save_pretrained('team_B_tiny_nanochat_model')
print('saved')

## Common computational cost benchmark

Every team reports the same computational quantities so the class can compare systems fairly.  

**Rules for the benchmark**

1. Use the final system your team would submit.
2. Run one warm-up report first. This avoids charging one-time CUDA/kernel initialization to the first measured case.
3. Time the same held-out test packets used by the other teams.
4. Report training and model-loading time separately from inference time.
5. Report both total parameters and trainable parameters. A deterministic template has 0; an off-the-shelf pretrained model has many total parameters but 0 parameters trained by this project; a LoRA system has a small trainable subset.
6. GPU memory is the peak PyTorch allocation during the benchmark when a GPU is used. The template system requires no GPU.

The cell below writes **`computational_metrics.csv`** with the same columns for all four teams. This file is one of the inputs to the classwide comparison next week.


In [ ]:
# ---- Standardized computational benchmark: Team B -------------------------
# Parameter storage is measured from the actual tensor dtypes, not assumed FP16.
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
parameter_memory_gb = parameter_bytes / 1024**3
hardware = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'

# Warm up once; do not count this report in timing.
_ = generate_tiny(normalized_test[0])
if device.type == 'cuda':
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

benchmark_start = time.perf_counter()
for p in normalized_test:
    _ = generate_tiny(p)
if device.type == 'cuda':
    torch.cuda.synchronize()
total_inference_seconds = time.perf_counter() - benchmark_start
peak_gpu_memory_gb = (torch.cuda.max_memory_allocated() / 1024**3) if device.type == 'cuda' else 0.0

n_reports = len(normalized_test)
seconds_per_report = total_inference_seconds / max(n_reports, 1)
reports_per_second = (n_reports / total_inference_seconds) if total_inference_seconds > 0 else float('inf')

metrics = pd.DataFrame([{
    'team': 'B',
    'system': 'Tiny specialized NanoChat-style model',
    'hardware': hardware,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'parameter_memory_gb': parameter_memory_gb,
    'peak_gpu_memory_gb': peak_gpu_memory_gb,
    'training_seconds': training_seconds,
    'model_load_seconds': model_load_seconds,
    'n_benchmark_reports': n_reports,
    'total_inference_seconds': total_inference_seconds,
    'seconds_per_report': seconds_per_report,
    'reports_per_second': reports_per_second,
    'estimated_seconds_per_1000_reports': seconds_per_report * 1000,
}])
metrics.to_csv('computational_metrics.csv', index=False)
display(metrics.T)
